# Case study 1 — leakage-free optimism ladder for transcriptomic breast-cancer metastasis classifiers
Reproduces every number of case study 1 in the manuscript from the public GEO series-matrix files.
Runtime: about 15 minutes on two CPU cores. Set `FAST = True` for a quick check (fewer repeats).

**Data**: GSE2034, GSE7390, GSE11121 and GSE2603 series-matrix files and the GPL96 annotation, downloaded below from NCBI GEO.
**Endpoints**: 5-year distant metastasis (GSE7390, GSE11121: t.dmfs/e.dmfs; GSE2603: authors' binary 5-year metastasis field;
patients censored before 5 years excluded) and bone relapse (GSE2034 bone-relapse flag; GSE2603 bone-metastasis event).

In [ ]:
import os, urllib.request, gzip, shutil
FAST = False
GEO_DIR = G = "geo"
os.makedirs(G, exist_ok=True)
URLS = {
 "GSE2034_series_matrix.txt.gz": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE2nnn/GSE2034/matrix/GSE2034_series_matrix.txt.gz",
 "GSE7390_series_matrix.txt.gz": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE7nnn/GSE7390/matrix/GSE7390_series_matrix.txt.gz",
 "GSE11121_series_matrix.txt.gz": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE11nnn/GSE11121/matrix/GSE11121_series_matrix.txt.gz",
 "GSE2603_series_matrix.txt.gz": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE2nnn/GSE2603/matrix/GSE2603_series_matrix.txt.gz",
 "GPL96.annot.gz": "https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPLnnn/GPL96/annot/GPL96.annot.gz",
}
for fn, url in URLS.items():
    if fn == "GPL96.annot.gz" and os.path.exists(f"{G}/GPL96_probe_symbol.tsv"): continue
    if not os.path.exists(f"{G}/{fn}"):
        print("downloading", fn); urllib.request.urlretrieve(url, f"{G}/{fn}")
# probe -> gene symbol table from the GPL96 annotation file
if not os.path.exists(f"{G}/GPL96_probe_symbol.tsv"):
    with gzip.open(f"{G}/GPL96.annot.gz", "rt", errors="replace") as fh:
        lines = fh.read().split("\n")
    h = next(i for i, l in enumerate(lines) if l.startswith("ID\t"))
    with open(f"{G}/GPL96_probe_symbol.tsv", "w") as out:
        out.write("probe\tsymbol\n")
        for l in lines[h + 1:]:
            f = l.split("\t")
            if len(f) > 2 and f[0] and not f[0].startswith("!"): out.write(f"{f[0]}\t{f[2]}\n")

## 1) Parse series matrices and construct endpoints

In [ ]:
"""Parse GEO series-matrix files into expression matrices and harmonised clinical tables.
Endpoint rules (binary, as in the manuscript):
  DM5  (5-year distant metastasis): 1 = distant event within 60 months; 0 = event-free with >= 60 months follow-up
        (or event after 60 months); patients censored before 60 months are excluded.
        GSE7390 / GSE11121: t.dmfs, e.dmfs (GEO characteristics). GSE2603: 'binary 5y met event' supplied by the authors.
  BONE (bone relapse, any time): GSE2034 'bone relapses (1=yes, 0=no)'; GSE2603 'bm event'.
Only primary tumours are kept (GSE2603 cell-line samples removed).
"""
import gzip, re, sys, os, numpy as np, pandas as pd



def read_matrix(acc):
    path = f"{GEO_DIR}/{acc}_series_matrix.txt.gz"
    meta, rows, in_table = {}, [], False
    with gzip.open(path, "rt", errors="replace") as fh:
        for line in fh:
            line = line.rstrip("\n")
            if line.startswith("!series_matrix_table_begin"): in_table = True; continue
            if line.startswith("!series_matrix_table_end"): break
            if in_table: rows.append(line.split("\t")); continue
            if line.startswith("!Sample_"):
                key = line.split("\t")[0]; vals = [v.strip('"') for v in line.split("\t")[1:]]
                meta.setdefault(key, []).append(vals)
    header = [h.strip('"') for h in rows[0]]
    df = pd.DataFrame([r[1:] for r in rows[1:]], index=[r[0].strip('"') for r in rows[1:]], columns=header[1:])
    df = df.apply(pd.to_numeric, errors="coerce")
    # characteristics -> dict of columns
    char = {}
    for vals in meta.get("!Sample_characteristics_ch1", []):
        for j, v in enumerate(vals):
            if ":" in v:
                k, x = v.split(":", 1); char.setdefault(k.strip(), [None] * len(vals))[j] = x.strip()
    clin = pd.DataFrame(char, index=header[1:])
    clin["title"] = meta["!Sample_title"][0]
    clin["source"] = meta["!Sample_source_name_ch1"][0]
    return df, clin

def num(s):
    return pd.to_numeric(s.replace({"--": np.nan, "?": np.nan, "NA": np.nan}), errors="coerce")

def dm5_from_times(t_months, e):
    y = pd.Series(np.nan, index=t_months.index)
    y[(t_months >= 60)] = 0     # followed for at least 5 y (event-free, or event after 5 y)
    y[(e == 1) & (t_months <= 60)] = 1
    return y

def load_all(accs=("GSE2034", "GSE7390", "GSE11121", "GSE2603")):
    X, C = {}, {}
    for acc in accs:
        if not os.path.exists(f"{GEO_DIR}/{acc}_series_matrix.txt.gz"):
            print(f"[warn] {acc} not available", file=sys.stderr); continue
        df, clin = read_matrix(acc)
        c = pd.DataFrame(index=clin.index); c["cohort"] = acc
        c["dm5"] = np.nan; c["bone"] = np.nan; c["size_cm"] = np.nan; c["grade"] = np.nan
        if acc == "GSE2034":
            c["bone"] = num(clin["bone relapses (1=yes, 0=no)"])
        elif acc == "GSE11121":
            t = num(clin["t.dmfs"]); e = num(clin["e.dmfs"])          # months
            c["dm5"] = dm5_from_times(t, e); c["size_cm"] = num(clin["size_in_cm"]); c["grade"] = num(clin["grade"])
        elif acc == "GSE7390":
            keys = {k.lower(): k for k in clin.columns}
            t = num(clin[keys["t.dmfs"]]) / 30.4375 if num(clin[keys["t.dmfs"]]).max() > 400 else num(clin[keys["t.dmfs"]])
            e = num(clin[keys["e.dmfs"]]); c["dm5"] = dm5_from_times(t, e)
            for k in clin.columns:
                if k.lower() in ("size", "size_in_cm"): c["size_cm"] = num(clin[k])
                if k.lower() == "grade": c["grade"] = num(clin[k])
        elif acc == "GSE2603":
            keep = clin["source"].str.contains("tumor|tumour|primary", case=False, regex=True) & ~clin["source"].str.contains("cell line|MDA|MCF", case=False, regex=True)
            clin = clin[keep]; df = df[clin.index]; c = c.loc[clin.index]
            c["dm5"] = num(clin["binary 5y met event"]); c["bone"] = num(clin["bm event"])
            c["size_cm"] = num(clin["tumor size (cm)"])
        X[acc], C[acc] = df, c
    return X, C

## 2) Gene-level compendium and patient flow (TRIPOD+AI)

In [ ]:
import numpy as np, pandas as pd, sys
"""Build the gene-level compendium (log2 scale) and the clinical table.
GSE2034, GSE11121, GSE2603: series-matrix MAS5 intensities -> log2(max(x, 1)).
GSE7390: series-matrix values are already log2 MAS5 (Bioconductor).
Genes: for every GPL96 probe annotated to exactly one gene symbol, the probe with the highest mean log2 expression
(computed on all 783 arrays, without any outcome) represents the gene -> 12,502 genes.
"""






gpl = pd.read_csv(f"{G}/GPL96_probe_symbol.tsv", sep="\t", keep_default_na=False)   # GPL96.annot: ID, Gene symbol
gpl = gpl[(gpl.symbol != "") & (~gpl.symbol.str.contains("///"))]
X, C = load_all(("GSE2034", "GSE11121", "GSE2603"))
df7, clin7 = read_matrix("GSE7390")
logs = [np.log2(X[a].clip(lower=1)) for a in X] + [df7]          # GSE7390 is already on the log2 scale
allx = pd.concat(logs, axis=1).loc[gpl.probe]
gpl = gpl.assign(mean=allx.mean(axis=1).values)
genes = gpl.sort_values("mean", ascending=False).drop_duplicates("symbol")          # label-free, all 783 arrays
order7 = [p for p in gpl.probe if p in set(genes.probe)]
sym = dict(zip(genes.probe, genes.symbol))
mats, clins = [], []
for acc in X:
    x = np.log2(X[acc].loc[order7].clip(lower=1)); x.index = [sym[p] for p in order7]
    mats.append(x); clins.append(C[acc])

# GSE7390 (values already log2 MAS5)
x7 = df7.loc[order7]; x7.index = [sym[p] for p in order7]
c7 = pd.DataFrame(index=x7.columns); c7["cohort"] = "GSE7390"
t = num(clin7["t.dmfs"]) / 30.4375; e = num(clin7["e.dmfs"])
c7["dm5"] = dm5_from_times(t, e); c7["bone"] = np.nan
c7["size_cm"] = num(clin7["size"]); c7["grade"] = num(clin7["grade"])
mats.append(x7); clins.append(c7)

expr = pd.concat(mats, axis=1)
clin = pd.concat(clins)
clin = clin.loc[expr.columns]
print("expression", expr.shape, "| clinical", clin.shape)
flow = []
for acc, c in clin.groupby("cohort", sort=False):
    flow.append(dict(cohort=acc, arrays=len(c), dm5_evaluable=int(c.dm5.notna().sum()), dm5_events=int((c.dm5 == 1).sum()),
                     dm5_excluded_censored=int(c.dm5.isna().sum()) if acc in ("GSE7390", "GSE11121", "GSE2603") else None,
                     bone_evaluable=int(c.bone.notna().sum()), bone_events=int((c.bone == 1).sum()),
                     size_grade_complete=int((c.size_cm.notna() & c.grade.notna()).sum())))
flow = pd.DataFrame(flow); print(flow.to_string(index=False))
np.savez_compressed(f"{G}/compendium_expr.npz", X=expr.T.values.astype(np.float32), genes=np.array(expr.index), samples=np.array(expr.columns))
clin.to_csv(f"{G}/compendium_clinical.csv")
flow.to_csv(f"{G}/patient_flow.csv", index=False)

## 3) Benchmark: leakage-free preprocessing, optimism ladder, sensitivity analyses, baselines, calibration, decision curves, stability

In [ ]:
"""Case study 1 (revision 3): leakage-free optimism ladder for transcriptomic metastasis classifiers.

Every preprocessing step that learns from data is fitted on training samples only:
  * quantile normalisation to a reference distribution estimated from the training samples (each sample mapped individually);
  * batch (cohort) location/scale adjustment estimated from training cohorts (empirical-Bayes ComBat, parametric);
    held-out samples from a cohort present in training are adjusted with that cohort's training estimates;
    an unseen cohort cannot be adjusted without using its own data, so under strict L1 it is only quantile-normalised
    and standardised with training statistics;
  * gene standardisation and feature selection (Mann-Whitney / AUC ranking, top 50 genes) on training samples only.
Rungs: L1 leave-one-cohort-out (strict) -> L2 repeated nested CV -> L3 = L2 with genes selected once on all samples ->
L4 = L3 on single 80/20 splits reporting the best of six models -> L5 resubstitution.
M2 sensitivity: L1 with (a) each test cohort centred/scaled on its own unlabelled data, (b) quantile normalisation + ComBat
fitted on all 783 arrays before splitting (the previous manuscript's pipeline).
"""
import numpy as np, pandas as pd, json, time, warnings, sys
from scipy.stats import norm, rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_sample_weight
warnings.filterwarnings("ignore")
SEED = 20260806

Z = np.load(f"{G}/compendium_expr.npz", allow_pickle=True)
XALL = Z["X"].astype(np.float64); GENES = Z["genes"]; SAMPLES = Z["samples"]
CLIN = pd.read_csv(f"{G}/compendium_clinical.csv", index_col=0).loc[SAMPLES]


# ---------------------------------------------------------------- preprocessing
def qn_reference(X):            # mean of sorted columns (samples are rows)
    return np.sort(X, axis=1).mean(axis=0)
def qn_apply(X, ref):
    r = np.argsort(np.argsort(X, axis=1), axis=1)
    return ref[r]

def combat_fit(X, batch):
    """Parametric empirical-Bayes ComBat (Johnson et al. 2007) without covariates. Returns per-batch gamma*, delta*
    and the pooled grand mean / variance, so the adjustment can be applied to new samples of a known batch."""
    b_ids = np.unique(batch); n, p = X.shape
    grand = X.mean(0); var_pooled = ((X - grand) ** 2).mean(0) + 1e-8
    S = (X - grand) / np.sqrt(var_pooled)
    params = {}
    gh = np.array([S[batch == b].mean(0) for b in b_ids]); dh = np.array([S[batch == b].var(0, ddof=1) + 1e-8 for b in b_ids])
    gbar = gh.mean(1); t2 = gh.var(1, ddof=1)
    a_prior = (2 * dh.var(1, ddof=1) + dh.mean(1) ** 2) / dh.var(1, ddof=1); b_prior = (dh.mean(1) * dh.var(1, ddof=1) + dh.mean(1) ** 3) / dh.var(1, ddof=1)
    for k, b in enumerate(b_ids):
        Sb = S[batch == b]; nb = Sb.shape[0]
        g_old, d_old = gh[k].copy(), dh[k].copy()
        for _ in range(50):
            g_new = (t2[k] * nb * gh[k] + d_old * gbar[k]) / (t2[k] * nb + d_old)
            sum2 = ((Sb - g_new) ** 2).sum(0)
            d_new = (0.5 * sum2 + b_prior[k]) / (nb / 2 + a_prior[k] - 1)
            if np.max(np.abs(g_new - g_old) / (np.abs(g_old) + 1e-8)) < 1e-4 and np.max(np.abs(d_new - d_old) / d_old) < 1e-4:
                g_old, d_old = g_new, d_new; break
            g_old, d_old = g_new, d_new
        params[b] = (g_old, d_old)
    return dict(grand=grand, var=var_pooled, params=params)
def combat_apply(X, batch, fit):
    S = (X - fit["grand"]) / np.sqrt(fit["var"]); out = S.copy()
    for b in np.unique(batch):
        m = batch == b
        if b in fit["params"]:
            g, d = fit["params"][b]; out[m] = (S[m] - g) / np.sqrt(d)
    return out * np.sqrt(fit["var"]) + fit["grand"]

class Prep:
    """mode: 'strict' (train-only QN + ComBat on training cohorts, unseen batches not adjusted),
             'test_self' (as strict, then each unseen test cohort is centred/scaled per gene on its own unlabelled data),
             'none' (no preprocessing; used when data were already harmonised globally)."""
    def __init__(self, mode="strict"): self.mode = mode
    def fit(self, X, batch):
        if self.mode == "none": return self
        self.ref = qn_reference(X); Xq = qn_apply(X, self.ref)
        self.cb = combat_fit(Xq, batch) if len(np.unique(batch)) > 1 else None
        Xc = combat_apply(Xq, batch, self.cb) if self.cb else Xq
        self.mu, self.sd = Xc.mean(0), Xc.std(0) + 1e-6
        return self
    def transform(self, X, batch, is_train=False):
        if self.mode == "none": return X
        Xq = qn_apply(X, self.ref)
        Xc = combat_apply(Xq, batch, self.cb) if self.cb else Xq
        if self.mode == "test_self" and not is_train:
            for b in np.unique(batch):
                if self.cb is None or b not in self.cb["params"]:
                    m = batch == b; Xc[m] = (Xc[m] - Xc[m].mean(0)) / (Xc[m].std(0) + 1e-6) * self.sd + self.mu
        return (Xc - self.mu) / self.sd

def global_harmonise(X, batch):          # previous manuscript: QN + ComBat on all arrays before any split
    Xq = qn_apply(X, qn_reference(X)); return combat_apply(Xq, batch, combat_fit(Xq, batch))

# ---------------------------------------------------------------- features, models, metrics
def auc_fast(score, y):
    r = rankdata(score); n1 = y.sum(); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
def select_genes(X, y, k=50):
    R = np.apply_along_axis(rankdata, 0, X); n1 = y.sum(); n0 = len(y) - n1
    auc = (R[y == 1].sum(0) - n1 * (n1 + 1) / 2) / (n1 * n0)
    return np.argsort(-np.abs(auc - 0.5))[:k]

def make_models():
    return {
        "Penalised LR": LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000),
        "SVM (RBF)": SVC(C=1.0, gamma="scale", probability=True, class_weight="balanced", random_state=SEED),
        "Random forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=3, max_features="sqrt", class_weight="balanced", n_jobs=2, random_state=SEED),
        "Gradient boosting": GradientBoostingClassifier(n_estimators=300 if FAST else 500, learning_rate=0.03, max_depth=3, subsample=0.8, max_features=0.6, random_state=SEED),
        "Hist. gradient boosting": HistGradientBoostingClassifier(max_iter=300 if FAST else 500, learning_rate=0.03, max_leaf_nodes=7, max_depth=3, l2_regularization=2.0, class_weight="balanced", random_state=SEED),
        "Neural network": MLPClassifier(hidden_layer_sizes=(64, 32), alpha=0.01, early_stopping=True, max_iter=500, random_state=SEED),
    }
def fit_predict(name, model, Xtr, ytr, Xte):
    if name == "Gradient boosting": model.fit(Xtr, ytr, sample_weight=compute_sample_weight("balanced", ytr))
    else: model.fit(Xtr, ytr)
    return model.predict_proba(Xte)[:, 1]

def delong(y, s):
    """AUC and DeLong variance (fast O(n log n) implementation)."""
    pos, neg = s[y == 1], s[y == 0]; m, n = len(pos), len(neg)
    tx = rankdata(pos); ty = rankdata(neg); tz = rankdata(np.concatenate([pos, neg]))
    auc = (tz[:m].sum() - m * (m + 1) / 2) / (m * n)
    v01 = (tz[:m] - tx) / n; v10 = 1 - (tz[m:] - ty) / m
    return auc, v01.var(ddof=1) / m + v10.var(ddof=1) / n, v01, v10
def delong_paired(y, s1, s2):
    a1, _, v01a, v10a = delong(y, s1); a2, _, v01b, v10b = delong(y, s2)
    m, n = (y == 1).sum(), (y == 0).sum()
    S01 = np.cov(np.vstack([v01a, v01b])); S10 = np.cov(np.vstack([v10a, v10b]))
    var = (S01[0, 0] + S01[1, 1] - 2 * S01[0, 1]) / m + (S10[0, 0] + S10[1, 1] - 2 * S10[0, 1]) / n
    zz = (a1 - a2) / np.sqrt(max(var, 1e-12)); return a1 - a2, 2 * norm.sf(abs(zz))
def random_effects(aucs, variances):
    """DerSimonian-Laird random-effects pooling of cohort AUCs on the logit scale."""
    a = np.clip(np.array(aucs), 1e-3, 1 - 1e-3); v = np.array(variances) / (a * (1 - a)) ** 2; th = np.log(a / (1 - a))
    w = 1 / v; fixed = np.sum(w * th) / w.sum(); Q = np.sum(w * (th - fixed) ** 2); k = len(a)
    tau2 = max(0.0, (Q - (k - 1)) / (w.sum() - (w ** 2).sum() / w.sum())) if k > 1 else 0.0
    ws = 1 / (v + tau2); est = np.sum(ws * th) / ws.sum(); se = np.sqrt(1 / ws.sum())
    inv = lambda x: 1 / (1 + np.exp(-x))
    return dict(auc=inv(est), lo=inv(est - 1.96 * se), hi=inv(est + 1.96 * se), tau2=tau2, I2=max(0, (Q - (k - 1)) / Q) * 100 if Q > 0 else 0.0)
def calib(y, p):
    lp = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    m = LogisticRegression(C=1e6, max_iter=1000).fit(lp[:, None], y); slope = m.coef_[0, 0]
    # calibration-in-the-large: intercept with the linear predictor as offset (Newton iterations)
    a = 0.0
    for _ in range(50):
        pr = 1 / (1 + np.exp(-(a + lp))); a += np.sum(y - pr) / np.sum(pr * (1 - pr))
    return slope, a
def net_benefit(y, p, thr):
    n = len(y); tp = np.sum((p >= thr) & (y == 1)); fp = np.sum((p >= thr) & (y == 0))
    return tp / n - fp / n * thr / (1 - thr)

# ---------------------------------------------------------------- task data
def task(name):
    col = "dm5" if name == "DM5" else "bone"
    m = CLIN[col].notna().values
    return XALL[m], CLIN[col].values[m].astype(int), CLIN.cohort.values[m], CLIN[m]

def run_task(tname, out):
    X, y, coh, clin = task(tname); cohorts = list(dict.fromkeys(coh)); rng = np.random.default_rng(SEED)
    models = list(make_models())
    res = dict(n=int(len(y)), events=int(y.sum()), cohorts={c: [int((coh == c).sum()), int(y[coh == c].sum())] for c in cohorts})
    t0 = time.time()

    # ---------- L1 variants: leave-one-cohort-out
    Xglob = global_harmonise(X, coh)
    L1 = {}
    for variant in ["strict", "test_self", "global"]:
        per = {mname: {} for mname in models}; preds = {mname: np.zeros(len(y)) for mname in models}; sels = {}
        for c in cohorts:
            te = coh == c; tr = ~te
            if variant == "global":
                Xtr_raw, Xte_raw = Xglob[tr], Xglob[te]
                mu, sd = Xtr_raw.mean(0), Xtr_raw.std(0) + 1e-6
                Xtr, Xte = (Xtr_raw - mu) / sd, (Xte_raw - mu) / sd
            else:
                pp = Prep(variant).fit(X[tr], coh[tr])
                Xtr = pp.transform(X[tr], coh[tr], is_train=True); Xte = pp.transform(X[te], coh[te])
            g = select_genes(Xtr, y[tr]); sels[c] = g
            for mname, mdl in make_models().items():
                p = fit_predict(mname, mdl, Xtr[:, g], y[tr], Xte[:, g]); preds[mname][te] = p
                a, v, _, _ = delong(y[te], p); per[mname][c] = (a, v)
        L1[variant] = dict(per=per, preds=preds, sels=sels)
        print(f"[{tname}] L1 {variant} done {time.time()-t0:.0f}s", flush=True)
    def summarize_L1(Lv):
        rows = {}
        for mname in models:
            cs = Lv["per"][mname]; re_ = random_effects([cs[c][0] for c in cohorts], [cs[c][1] for c in cohorts])
            rows[mname] = dict(per_cohort={c: dict(auc=cs[c][0], lo=cs[c][0] - 1.96 * np.sqrt(cs[c][1]), hi=cs[c][0] + 1.96 * np.sqrt(cs[c][1])) for c in cohorts},
                               mean_auc=float(np.mean([cs[c][0] for c in cohorts])), meta=re_)
        return rows
    res["L1"] = {v: summarize_L1(L1[v]) for v in L1}

    # ---------- L2 / L3: repeated stratified 5-fold CV (fold-internal preprocessing); L3 selects genes once on all samples
    reps = 2 if FAST else 4
    pp_all = Prep("strict").fit(X, coh); Xall_std = pp_all.transform(X, coh, is_train=True)
    g_leaky = select_genes(Xall_std, y)
    L23 = {"L2": {m: [] for m in models}, "L3": {m: [] for m in models}}
    for r in range(reps):
        skf = StratifiedKFold(5, shuffle=True, random_state=SEED + r)
        for tr_i, te_i in skf.split(X, y):
            pp = Prep("strict").fit(X[tr_i], coh[tr_i])
            Xtr = pp.transform(X[tr_i], coh[tr_i], is_train=True); Xte = pp.transform(X[te_i], coh[te_i])
            g_nested = select_genes(Xtr, y[tr_i])
            for rung, g in [("L2", g_nested), ("L3", g_leaky)]:
                for mname, mdl in make_models().items():
                    p = fit_predict(mname, mdl, Xtr[:, g], y[tr_i], Xte[:, g])
                    L23[rung][mname].append(auc_fast(p, y[te_i]))
        print(f"[{tname}] L2/L3 repeat {r+1}/{reps} {time.time()-t0:.0f}s", flush=True)
    # ---------- L4: leaky selection, single 80/20 split, best of six models (20 splits)
    L4 = []; L4_all = {m: [] for m in models}
    for s in range(10 if FAST else 20):
        tr_i, te_i = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=SEED + 100 + s)
        best = 0
        for mname, mdl in make_models().items():
            p = fit_predict(mname, mdl, Xall_std[tr_i][:, g_leaky], y[tr_i], Xall_std[te_i][:, g_leaky])
            a = auc_fast(p, y[te_i]); L4_all[mname].append(a); best = max(best, a)
        L4.append(best)
    # ---------- L5: resubstitution
    L5 = {mname: auc_fast(fit_predict(mname, mdl, Xall_std[:, g_leaky], y, Xall_std[:, g_leaky]), y) for mname, mdl in make_models().items()}
    print(f"[{tname}] L4/L5 done {time.time()-t0:.0f}s", flush=True)

    def dist(v): v = np.array(v); return dict(mean=float(v.mean()), lo=float(np.percentile(v, 2.5)), hi=float(np.percentile(v, 97.5)), n=int(v.size))
    res["L2"] = {m: dist(L23["L2"][m]) for m in models}; res["L3"] = {m: dist(L23["L3"][m]) for m in models}
    res["L4_best"] = dist(L4); res["L4_models"] = {m: dist(L4_all[m]) for m in models}; res["L5"] = L5
    ladder = dict(L1_strict=float(np.mean([res["L1"]["strict"][m]["mean_auc"] for m in models])),
                  L1_test_self=float(np.mean([res["L1"]["test_self"][m]["mean_auc"] for m in models])),
                  L1_global=float(np.mean([res["L1"]["global"][m]["mean_auc"] for m in models])),
                  L2=float(np.mean([res["L2"][m]["mean"] for m in models])), L3=float(np.mean([res["L3"][m]["mean"] for m in models])),
                  L4=res["L4_best"]["mean"], L5=float(np.mean(list(L5.values()))))
    # uncertainty for the ladder: spread across models (L1, L5) and across folds x models (L2-L4)
    ladder_ci = dict(L1_strict=[float(min(res["L1"]["strict"][m]["mean_auc"] for m in models)), float(max(res["L1"]["strict"][m]["mean_auc"] for m in models))],
                     L2=[float(np.percentile(sum(L23["L2"].values(), []), 2.5)), float(np.percentile(sum(L23["L2"].values(), []), 97.5))],
                     L3=[float(np.percentile(sum(L23["L3"].values(), []), 2.5)), float(np.percentile(sum(L23["L3"].values(), []), 97.5))],
                     L4=[res["L4_best"]["lo"], res["L4_best"]["hi"]], L5=[float(min(L5.values())), float(max(L5.values()))])
    res["ladder"] = ladder; res["ladder_range"] = ladder_ci

    # ---------- model comparisons under strict L1: per cohort DeLong vs penalised LR, Holm-adjusted
    comps = []
    for mname in models:
        if mname == "Penalised LR": continue
        for c in cohorts:
            te = coh == c
            d, pval = delong_paired(y[te], L1["strict"]["preds"]["Penalised LR"][te], L1["strict"]["preds"][mname][te])
            comps.append(dict(model=mname, cohort=c, diff_LR_minus_model=d, p=pval))
    comps = pd.DataFrame(comps).sort_values("p").reset_index(drop=True); k = len(comps)
    comps["p_holm"] = np.minimum(1, np.maximum.accumulate((k - np.arange(k)) * comps.p))
    res["delong"] = comps.to_dict(orient="records")

    # ---------- baselines under strict L1: unfitted proliferation metagene, random 50-gene sets (penalised LR)
    prolif = ["BIRC5", "CCNB1", "CDC20", "CEP55", "NDC80", "MKI67", "PTTG1", "RRM2", "TYMS", "UBE2C"]
    pidx = [int(np.where(GENES == gname)[0][0]) for gname in prolif if gname in set(GENES)]
    base = dict(prolif={}, random={}, lr_learned={})
    n_rand = 20 if FAST else 60
    rand_sets = [rng.choice(len(GENES), 50, replace=False) for _ in range(n_rand)]
    for c in cohorts:
        te = coh == c; tr = ~te
        pp = Prep("strict").fit(X[tr], coh[tr]); Xtr = pp.transform(X[tr], coh[tr], is_train=True); Xte = pp.transform(X[te], coh[te])
        base["prolif"][c] = auc_fast(Xte[:, pidx].mean(1), y[te])
        ra = []
        for gs in rand_sets:
            m = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000).fit(Xtr[:, gs], y[tr]); ra.append(auc_fast(m.predict_proba(Xte[:, gs])[:, 1], y[te]))
        base["random"][c] = float(np.mean(ra))
        base["lr_learned"][c] = res["L1"]["strict"]["Penalised LR"]["per_cohort"][c]["auc"]
    res["baselines"] = {k2: dict(per_cohort=v, mean=float(np.mean(list(v.values())))) for k2, v in base.items()}
    res["prolif_genes_used"] = [gname for gname in prolif if gname in set(GENES)]

    # ---------- calibration (strict L1, per held-out cohort) and decision curve (pooled held-out predictions)
    cal = {}
    for mname in models:
        cal[mname] = {c: dict(zip(["slope", "intercept"], calib(y[coh == c], L1["strict"]["preds"][mname][coh == c]))) for c in cohorts}
    res["calibration"] = cal
    thr = np.round(np.arange(0.05, 0.61, 0.025), 3); prev = y.mean()
    dca = {}
    for mname in models:
        p = L1["strict"]["preds"][mname]
        nb = [net_benefit(y, p, t) for t in thr]; nb_all = [prev - (1 - prev) * t / (1 - t) for t in thr]
        useful = [float(t) for t, a, b in zip(thr, nb, nb_all) if a > max(b, 0) + 1e-9]
        dca[mname] = dict(useful_thresholds=useful, nb=nb)
    res["dca"] = dict(prevalence=float(prev), thresholds=thr.tolist(), treat_all=[prev - (1 - prev) * t / (1 - t) for t in thr], models=dca)

    # ---------- signature stability: 50 subsamples of 80% without replacement, top-50 genes after strict preprocessing
    sigs = []
    for s in range(20 if FAST else 50):
        idx, _ = train_test_split(np.arange(len(y)), train_size=0.8, stratify=y, random_state=SEED + 500 + s)
        pp = Prep("strict").fit(X[idx], coh[idx]); Xs = pp.transform(X[idx], coh[idx], is_train=True)
        sigs.append(set(select_genes(Xs, y[idx]).tolist()))
    jac = [len(a & b) / len(a | b) for i, a in enumerate(sigs) for b in sigs[i + 1:]]
    freq = pd.Series([gg for s_ in sigs for gg in s_]).value_counts() / len(sigs)
    res["stability"] = dict(scheme="80% stratified subsamples without replacement", n=len(sigs), jaccard_mean=float(np.mean(jac)),
                            jaccard_sd=float(np.std(jac)), distinct=int(len(freq)), in_80pct=int((freq >= 0.8).sum()), in_all=int((freq >= 1).sum()))

    # ---------- clinical model (size + grade) vs transcriptomic models on the SAME complete-case patients (DM5 only)
    if tname == "DM5":
        cc = clin.size_cm.notna().values & clin.grade.notna().values
        cc_coh = [c for c in cohorts if (cc & (coh == c)).sum() > 0]
        clin_res = {}
        for c in cc_coh:
            te = cc & (coh == c); tr = cc & (coh != c)
            Ztr = clin[["size_cm", "grade"]].values[tr]; Zte = clin[["size_cm", "grade"]].values[te]
            mu, sd = Ztr.mean(0), Ztr.std(0)
            m = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000).fit((Ztr - mu) / sd, y[tr])
            a_clin = auc_fast(m.predict_proba((Zte - mu) / sd)[:, 1], y[te])
            # transcriptomic models trained on the same complete-case training patients
            pp = Prep("strict").fit(X[tr], coh[tr]); Xtr = pp.transform(X[tr], coh[tr], is_train=True); Xte = pp.transform(X[te], coh[te])
            g = select_genes(Xtr, y[tr]); a_tx = {}
            for mname, mdl in make_models().items(): a_tx[mname] = auc_fast(fit_predict(mname, mdl, Xtr[:, g], y[tr], Xte[:, g]), y[te])
            clin_res[c] = dict(n=int(te.sum()), events=int(y[te].sum()), clinical=a_clin, transcriptomic=a_tx)
        res["clinical_vs_tx"] = clin_res
    print(f"[{tname}] finished {time.time()-t0:.0f}s", flush=True)
    return res

if True:
    tasks = ["DM5", "BONE"]
    out = {}
    for t in tasks:
        out[t] = run_task(t, out)
        json.dump(out, open(f"results_{'fast' if FAST else 'full'}.json", "w"), indent=1, default=float)
    print("saved")